In [1]:
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import joblib
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [2]:
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

x_train_raw = joblib.load('x_train.joblib')
le = joblib.load('label_encoder.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

x_train = torch.tensor(x_train_raw.toarray(), dtype=torch.float32)

In [3]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = ' '.join(tokens)
    text = [ps.stem(word) for word in tokens.split()]
    return ' '.join(text)

In [4]:
class SentimentModel(nn.Module): 
    def __init__(self):
        super(SentimentModel, self).__init__()
        self.f1 = nn.Linear(150, 50)   
        self.f2 = nn.Linear(50, 20)    
        self.f3 = nn.Linear(20, 4)     

    def forward(self, x):
        x = torch.relu(self.f1(x))
        x = torch.relu(self.f2(x))
        return self.f3(x) 

In [5]:

model = SentimentModel()  
model.load_state_dict(torch.load("sentiment_model.pth", map_location="cpu")) 

<All keys matched successfully>

In [6]:
model.eval()  
feature_names = tfidf.get_feature_names_out()

def predict_func(x_numpy):
    x_tensor = torch.tensor(x_numpy, dtype=torch.float32)
    with torch.no_grad():
        outputs = model(x_tensor)
    return outputs.numpy()

background_numpy = x_train_raw.toarray()[:100]
explainer = shap.Explainer(predict_func, background_numpy)

In [7]:
raw_sentence = "The phone has an amazing screen, but the battery life is terrible."
clean_sentence = preprocess_text(raw_sentence)

test_vector_dense = tfidf.transform([clean_sentence]).toarray()

with torch.no_grad():
    predictions = model(torch.tensor(test_vector_dense, dtype=torch.float32))
    predicted_class_idx = torch.argmax(predictions, dim=1).item()
    predicted_label = le.inverse_transform([predicted_class_idx])

print(f"Preprocessed Sentence: '{clean_sentence}'")
print(f"Predicted Class Index: {predicted_class_idx} ({predicted_label})\n")

Preprocessed Sentence: 'phone amaz screen batteri life terribl'
Predicted Class Index: 1 (['Normal'])



In [8]:

shap_values = explainer(test_vector_dense)

word_scores = shap_values.values[0, :, predicted_class_idx]

activated_words = []
for i, score in enumerate(word_scores):
    if test_vector_dense[0, i] > 0:
        activated_words.append((feature_names[i], score))

df_importance = pd.DataFrame(activated_words, columns=['Stemmed Word', 'SHAP Impact Score'])
df_importance = df_importance.sort_values(by='SHAP Impact Score', ascending=False)

print(df_importance.to_string(index=False))


Stemmed Word  SHAP Impact Score
        life          -1.482697


In [9]:
with open("shap_explainer.pkl", "wb") as f:
    explainer.save(f)

print("Explainer saved successfully using SHAP's native method.")

Explainer saved successfully using SHAP's native method.
